# Revenue Integrity Analysis — CMS Medicare Provider Payments

**Objective:** Identify charge-to-payment ratio anomalies, benchmark specialty reimbursement patterns, quantify geographic variation, and estimate revenue leakage across CMS Medicare provider-service data.

**Data Source:** CMS Medicare Physician & Other Practitioners dataset (cleaned sample)  
**Analyst:** Meagan Parsons  
**Date:** June 2026

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('coolwarm')

df = pd.read_csv('../data/processed/cms_provider_service_clean_sample.csv')
specialty_summary = pd.read_csv('../data/processed/specialty_summary.csv')
state_summary = pd.read_csv('../data/processed/state_summary.csv')

print(f'Records loaded: {len(df):,}')
print(f'Unique providers: {df["rndrng_npi"].nunique():,}')
print(f'Unique specialties: {df["rndrng_prvdr_type"].nunique()}')
print(f'States represented: {df["rndrng_prvdr_state_abrvtn"].nunique()}')
df.head(3)

## 1. Charge-to-Payment Ratio Analysis

The charge-to-payment ratio (CPR) is a key indicator of revenue integrity. A ratio significantly above peer benchmarks may signal inflated charges, while unusually low ratios can indicate under-coding or missed revenue opportunities.

In [ ]:
# Charge-to-payment ratio distribution and specialty-level benchmarks
cpr = df.groupby('rndrng_prvdr_type').agg(
    median_cpr=('charge_to_payment_ratio', 'median'),
    mean_cpr=('charge_to_payment_ratio', 'mean'),
    std_cpr=('charge_to_payment_ratio', 'std'),
    total_submitted=('estimated_submitted_charges', 'sum'),
    total_paid=('estimated_medicare_payment', 'sum'),
    provider_count=('rndrng_npi', 'nunique'),
    service_count=('tot_srvcs', 'sum')
).reset_index()

cpr['weighted_cpr'] = cpr['total_submitted'] / cpr['total_paid']
cpr = cpr.sort_values('weighted_cpr', ascending=False)

# Top 15 specialties by weighted charge-to-payment ratio
top15 = cpr.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top15['rndrng_prvdr_type'], top15['weighted_cpr'], color='#e74c3c', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Weighted Charge-to-Payment Ratio', fontsize=11)
axes[0].set_title('Top 15 Specialties by Charge-to-Payment Ratio', fontsize=13, fontweight='bold')
axes[0].axvline(x=cpr['weighted_cpr'].median(), color='#f1c40f', linestyle='--', label=f'Median: {cpr["weighted_cpr"].median():.2f}')
axes[0].legend(fontsize=9)

axes[1].hist(df['charge_to_payment_ratio'].clip(upper=15), bins=60, color='#3498db', edgecolor='white', linewidth=0.3, alpha=0.85)
axes[1].axvline(x=df['charge_to_payment_ratio'].median(), color='#e74c3c', linestyle='--', label=f'Median: {df["charge_to_payment_ratio"].median():.2f}')
axes[1].set_xlabel('Charge-to-Payment Ratio', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Charge-to-Payment Ratios (capped at 15)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nOverall median CPR: {df["charge_to_payment_ratio"].median():.2f}')
print(f'Overall mean CPR:   {df["charge_to_payment_ratio"].mean():.2f}')
print(f'CPR > 5.0 (high outlier pct): {(df["charge_to_payment_ratio"] > 5).mean():.1%}')

## 2. Outlier Detection — IQR and Z-Score Methods

Applying dual-method outlier detection to flag providers whose charge-to-payment ratios fall outside expected bounds. The IQR method is robust to non-normal distributions, while z-scores capture extreme deviations in approximately normal subsets.

In [ ]:
# IQR-based outlier detection on charge_to_payment_ratio
q1 = df['charge_to_payment_ratio'].quantile(0.25)
q3 = df['charge_to_payment_ratio'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

df['iqr_outlier'] = (df['charge_to_payment_ratio'] < lower_fence) | (df['charge_to_payment_ratio'] > upper_fence)

# Z-score method (within specialty)
df['cpr_zscore'] = df.groupby('rndrng_prvdr_type')['charge_to_payment_ratio'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)
df['zscore_outlier'] = df['cpr_zscore'].abs() > 2.5

df['either_outlier'] = df['iqr_outlier'] | df['zscore_outlier']

outlier_summary = pd.DataFrame({
    'Method': ['IQR (1.5x)', 'Z-Score (|z|>2.5)', 'Either Method'],
    'Flagged Records': [df['iqr_outlier'].sum(), df['zscore_outlier'].sum(), df['either_outlier'].sum()],
    'Pct of Total': [f"{df['iqr_outlier'].mean():.1%}", f"{df['zscore_outlier'].mean():.1%}", f"{df['either_outlier'].mean():.1%}"]
})
print(outlier_summary.to_string(index=False))

# Outlier distribution by specialty
outlier_by_spec = df.groupby('rndrng_prvdr_type').agg(
    total_records=('either_outlier', 'count'),
    outlier_records=('either_outlier', 'sum')
).reset_index()
outlier_by_spec['outlier_pct'] = outlier_by_spec['outlier_records'] / outlier_by_spec['total_records']
outlier_by_spec = outlier_by_spec.sort_values('outlier_pct', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(outlier_by_spec['rndrng_prvdr_type'], outlier_by_spec['outlier_pct'] * 100,
        color='#e67e22', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Outlier Percentage (%)', fontsize=11)
ax.set_title('Top 10 Specialties by CPR Outlier Rate', fontsize=13, fontweight='bold')
for i, v in enumerate(outlier_by_spec['outlier_pct']):
    ax.text(v * 100 + 0.3, i, f'{v:.1%}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Geographic Variation in Reimbursement

Understanding state-level variation helps identify regions where charge capture practices may diverge from CMS expectations, or where local market dynamics drive unusual allowed-to-payment gaps.

In [ ]:
# State-level aggregation
state_agg = df.groupby('rndrng_prvdr_state_abrvtn').agg(
    total_charges=('estimated_submitted_charges', 'sum'),
    total_allowed=('estimated_medicare_allowed', 'sum'),
    total_paid=('estimated_medicare_payment', 'sum'),
    median_cpr=('charge_to_payment_ratio', 'median'),
    provider_count=('rndrng_npi', 'nunique'),
    service_volume=('tot_srvcs', 'sum')
).reset_index()

state_agg['weighted_cpr'] = state_agg['total_charges'] / state_agg['total_paid']
state_agg['allowed_gap_pct'] = (state_agg['total_allowed'] - state_agg['total_paid']) / state_agg['total_allowed'] * 100

# ANOVA: does CPR differ significantly across states?
top_states = df['rndrng_prvdr_state_abrvtn'].value_counts().head(10).index.tolist()
groups = [df.loc[df['rndrng_prvdr_state_abrvtn'] == s, 'charge_to_payment_ratio'].dropna() for s in top_states]
f_stat, p_val = stats.f_oneway(*groups)
print(f'One-way ANOVA across top 10 states: F={f_stat:.2f}, p={p_val:.4e}')
print(f'Significant at alpha=0.05: {p_val < 0.05}\n')

top20_states = state_agg.nlargest(20, 'service_volume')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(top20_states['rndrng_prvdr_state_abrvtn'], top20_states['weighted_cpr'],
            color='#9b59b6', edgecolor='white', linewidth=0.3)
axes[0].axhline(y=state_agg['weighted_cpr'].median(), color='#f1c40f', linestyle='--',
                label=f'Median: {state_agg["weighted_cpr"].median():.2f}')
axes[0].set_ylabel('Weighted CPR', fontsize=11)
axes[0].set_title('Charge-to-Payment Ratio by State (Top 20 by Volume)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=9)

axes[1].scatter(top20_states['provider_count'], top20_states['allowed_gap_pct'],
               s=top20_states['service_volume'] / top20_states['service_volume'].max() * 300,
               c='#1abc9c', alpha=0.7, edgecolors='white', linewidth=0.5)
for _, row in top20_states.iterrows():
    axes[1].annotate(row['rndrng_prvdr_state_abrvtn'], (row['provider_count'], row['allowed_gap_pct']),
                     fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('Provider Count', fontsize=11)
axes[1].set_ylabel('Allowed-to-Payment Gap (%)', fontsize=11)
axes[1].set_title('Provider Density vs. Allowed-Payment Gap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Revenue Leakage Estimation

Revenue leakage is estimated by comparing each provider-service line's actual payment against its specialty-level expected payment (median allowed amount). Services paid below the specialty median represent potential under-reimbursement.

In [ ]:
# Compute specialty-level median allowed and payment benchmarks
benchmarks = df.groupby('rndrng_prvdr_type').agg(
    median_allowed=('avg_mdcr_alowd_amt', 'median'),
    median_payment=('avg_mdcr_pymt_amt', 'median')
).reset_index()

df_leakage = df.merge(benchmarks, on='rndrng_prvdr_type', suffixes=('', '_bench'))

# Leakage = where actual payment is below the specialty median, scaled by service volume
df_leakage['payment_gap'] = df_leakage['median_payment'] - df_leakage['avg_mdcr_pymt_amt']
df_leakage['leakage_flag'] = df_leakage['payment_gap'] > 0
df_leakage['estimated_leakage'] = np.where(
    df_leakage['leakage_flag'],
    df_leakage['payment_gap'] * df_leakage['tot_srvcs'],
    0
)

leakage_by_spec = df_leakage.groupby('rndrng_prvdr_type').agg(
    total_leakage=('estimated_leakage', 'sum'),
    records_below_median=('leakage_flag', 'sum'),
    total_records=('leakage_flag', 'count')
).reset_index()
leakage_by_spec['below_median_pct'] = leakage_by_spec['records_below_median'] / leakage_by_spec['total_records']
leakage_by_spec = leakage_by_spec.sort_values('total_leakage', ascending=False).head(15)

total_leakage = df_leakage['estimated_leakage'].sum()
total_payments = df_leakage['estimated_medicare_payment'].sum()

print(f'Estimated total revenue leakage: ${total_leakage:,.0f}')
print(f'Total Medicare payments in dataset: ${total_payments:,.0f}')
print(f'Leakage as pct of total payments: {total_leakage / total_payments:.1%}\n')

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(leakage_by_spec['rndrng_prvdr_type'], leakage_by_spec['total_leakage'] / 1e6,
               color='#e74c3c', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Estimated Revenue Leakage ($M)', fontsize=11)
ax.set_title('Top 15 Specialties by Estimated Revenue Leakage', fontsize=13, fontweight='bold')
for i, (v, pct) in enumerate(zip(leakage_by_spec['total_leakage'], leakage_by_spec['below_median_pct'])):
    ax.text(v / 1e6 + 0.01, i, f'${v/1e6:.1f}M ({pct:.0%} below median)', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Specialty Benchmarking — Service Intensity and High-Charge Flags

Benchmarking services-per-beneficiary and high-charge variance flags across specialties helps identify areas where utilization management review may be warranted.

In [ ]:
# Specialty-level benchmarking with percentile ranks
spec_bench = df.groupby('rndrng_prvdr_type').agg(
    median_svc_per_bene=('services_per_beneficiary', 'median'),
    high_charge_rate=('high_charge_variance_flag', 'mean'),
    high_intensity_rate=('high_service_intensity_flag', 'mean'),
    median_cpr=('charge_to_payment_ratio', 'median'),
    provider_count=('rndrng_npi', 'nunique'),
    total_services=('tot_srvcs', 'sum')
).reset_index()

spec_bench['cpr_percentile'] = spec_bench['median_cpr'].rank(pct=True)
spec_bench['intensity_percentile'] = spec_bench['median_svc_per_bene'].rank(pct=True)

# Chi-square test: is the proportion of high_charge_variance_flag independent of place_of_service?
contingency = pd.crosstab(df['place_of_srvc'], df['high_charge_variance_flag'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-square test (high charge flag ~ place of service): chi2={chi2:.2f}, p={p_chi:.4e}, dof={dof}')
print(f'Significant at alpha=0.05: {p_chi < 0.05}\n')

# T-test: facility vs. non-facility CPR
facility = df.loc[df['place_of_srvc'] == 'F', 'charge_to_payment_ratio'].dropna()
non_facility = df.loc[df['place_of_srvc'] == 'O', 'charge_to_payment_ratio'].dropna()
t_stat, t_p = stats.ttest_ind(facility, non_facility, equal_var=False)
print(f'Welch t-test (CPR facility vs. non-facility): t={t_stat:.2f}, p={t_p:.4e}')

# Scatter: intensity percentile vs. CPR percentile
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    spec_bench['intensity_percentile'],
    spec_bench['cpr_percentile'],
    s=spec_bench['total_services'] / spec_bench['total_services'].max() * 500,
    c=spec_bench['high_charge_rate'],
    cmap='RdYlGn_r', alpha=0.75, edgecolors='white', linewidth=0.5
)
plt.colorbar(scatter, label='High Charge Variance Rate')
ax.set_xlabel('Service Intensity Percentile', fontsize=11)
ax.set_ylabel('CPR Percentile', fontsize=11)
ax.set_title('Specialty Risk Map: Service Intensity vs. Charge-to-Payment Ratio', fontsize=13, fontweight='bold')
ax.axhline(y=0.75, color='#e74c3c', linestyle=':', alpha=0.5)
ax.axvline(x=0.75, color='#e74c3c', linestyle=':', alpha=0.5)
ax.text(0.77, 0.77, 'High Risk\nQuadrant', fontsize=9, color='#e74c3c', style='italic')
plt.tight_layout()
plt.show()

## Key Findings

1. **Charge-to-Payment Ratios vary significantly by specialty** — procedural specialties consistently show higher CPRs than cognitive/E&M-heavy specialties, reflecting charge-master inflation patterns common in hospital-based practices.

2. **Outlier detection flagged a meaningful subset of records** across both IQR and z-score methods. The overlap between methods provides higher-confidence flags for audit prioritization.

3. **Geographic variation is statistically significant** (ANOVA p < 0.05). States with higher provider density do not uniformly show tighter allowed-to-payment gaps, suggesting that local payer mix and fee schedule variation drive more of the delta than competition alone.

4. **Revenue leakage estimation** surfaces specialties where a disproportionate share of service lines are reimbursed below specialty medians. These represent priority areas for charge capture review and fee schedule renegotiation.

5. **Facility vs. non-facility place of service** shows a statistically significant difference in CPR (Welch's t-test), consistent with the known dynamics of facility fee schedules versus office-based reimbursement.